# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Clinicopathological dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, following Croissant schema conventions and referencing entities by their `@id`s.

### Dataset Source
The dataset metadata and schema are provided by the following Croissant JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

We'll use `mlcroissant` to load the dataset's metadata and inspect its structure. This will allow us to programmatically access record set, field, and column IDs using the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Let's list all record sets and their field and column `@id`s present in the dataset. This helps us know what data is available and which `@id`s to use for extraction and analysis.

In [ ]:
# List all record sets, fields, and column @id's

record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
overview = {}
for rs in record_sets:
    print(f"Record set name: {rs.name}, @id: {rs.id}")
    fields = rs.fields
    columns = rs.columns
    overview[rs.id] = {
        'name': rs.name,
        'fields': [field.id for field in fields],
        'columns': [col.id for col in columns]
    }
    print(f"  Fields: {overview[rs.id]['fields']}")
    print(f"  Columns: {overview[rs.id]['columns']}")
    print()

## 3. Data Extraction

We'll now extract records from each record set into a pandas DataFrame, referencing record sets by their `@id` as required. Feel free to adjust which record sets to load based on the overview above.

In [ ]:
# Prepare to extract all (or a subset) of the record sets by @id
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")
    print()

Below, let's pick the main tabular record set (if there's just one, or select the most substantial by browsing the list above), reference its `@id`, and display the first 5 rows for preview.

*If there are multiple record sets, you may adjust `main_record_set_id` as appropriate*.

In [ ]:
# Choose a main record set for analysis from dataframes loaded above
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]  # Use the first record set @id loaded above
    print(f"Using record set: {main_record_set_id}")
    df = dataframes[main_record_set_id]
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Now, let's apply some basic data processing steps. We'll select a numeric field by its `@id` for filtering and normalization, and also try to group by a categorical field.

You can reference the DataFrame columns printed above (which reflect the Croissant `@id`s of the fields/columns) to select a numeric and group field.

In [ ]:
# Choose a numeric field and a group field by examining the columns above
# Replace these with the actual @id's (column names) found for your dataset

# Example selection: (Replace with your dataset's true @id's!)
numeric_field_id = None
group_field_id = None

if 'schema:AgeAtDiagnosis' in df.columns:
    numeric_field_id = 'schema:AgeAtDiagnosis'
elif len(df.select_dtypes(include=['float', 'int'])):
    numeric_field_id = df.select_dtypes(include=['float', 'int']).columns[0]

if 'schema:Sex' in df.columns:
    group_field_id = 'schema:Sex'
elif len(df.select_dtypes(include=['object'])):
    group_field_id = df.select_dtypes(include=['object']).columns[0]

if numeric_field_id:
    print(f"Analyzing numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 0
    # Remove outliers (e.g. values above 99th percentile)
    upper = df[numeric_field_id].quantile(0.99)
    filtered_df = df[df[numeric_field_id] < upper]
    print(f"Filtered to records with {numeric_field_id} < P99 ({upper:.2f})")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found suitable for EDA.")

## 5. Visualization

Let's plot the distribution of the selected numeric field, and if grouped, compare group means.

*If matplotlib/seaborn are not installed, uncomment the install lines below.*

In [ ]:
# !pip install matplotlib seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² clinicopathological dataset using the Croissant schema and the `mlcroissant` package. We:
- Loaded dataset metadata and programmatically inspected record sets by their `@id`
- Loaded structured records into DataFrames using Croissant `@id` references
- Performed basic filtering, normalization, and grouping by referenced fields
- Visualized data distributions and group differences

You can extend this workflow to perform more advanced modeling or join additional record sets as required. Always refer to schema and field `@id`s to remain consistent and reproducible.